# Explore the bakery datasets

This notebook inventories the sales, weather, and Kiel Week source files. It measures date coverage, missing values, and what happens when the files are joined. Historical turnover plots and correlations use the training period in the project canvas.

## Load the files

The notebook reads the [course CSVs](https://raw.githubusercontent.com/opencampus-sh/intro-to-data-science-and-ml/main/data/bakery/README.md) from GitHub by default. For an offline run, set the `BAKERY_DATA_DIR` environment variable to a folder containing an unmodified copy of all three course CSVs before starting the notebook. `Datum` means date and is read as a date in each table.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

BASE_URL = (
    "https://raw.githubusercontent.com/opencampus-sh/"
    "intro-to-data-science-and-ml/main/data/bakery/"
)
local_data_dir = os.environ.get("BAKERY_DATA_DIR")
sources = {
    "Sales": "umsatzdaten_gekuerzt.csv",
    "Weather": "wetter.csv",
    "Kiel Week": "kiwo.csv",
}

frames = {}
for name, filename in sources.items():
    csv_path = Path(local_data_dir) / filename if local_data_dir else BASE_URL + filename
    dtypes = {"id": "string"} if name == "Sales" else None
    frames[name] = pd.read_csv(csv_path, parse_dates=["Datum"], dtype=dtypes)

sales = frames["Sales"]
weather = frames["Weather"]
kiel_week = frames["Kiel Week"]
product_names = {
    1: "Bread", 2: "Rolls", 3: "Croissants",
    4: "Confectionery", 5: "Cakes", 6: "Seasonal bread",
}
weekdays = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

## Source inventory and calendar coverage

Sales have one row per date and recorded product group. Weather has one row per observation date. Kiel Week lists event dates, so days between events are outside the file’s scope. For sales and weather, a gap means that no row exists for that date. The first and last recorded dates describe coverage at the edges of each file.

In [ ]:
keys = {
    "Sales": ["Datum", "Warengruppe"],
    "Weather": ["Datum"],
    "Kiel Week": ["Datum"],
}
inventory = []
absent_dates = {}

for name, frame in frames.items():
    dates = pd.DatetimeIndex(frame["Datum"].dropna().unique())
    first, last = dates.min(), dates.max()
    calendar = pd.date_range(first, last)
    if name != "Kiel Week":
        absent_dates[name] = calendar.difference(dates)

    inventory.append({
        "Source": name,
        "Rows": len(frame),
        "Columns": len(frame.columns),
        "Key": ", ".join(keys[name]),
        "Rows with repeated key": int(frame.duplicated(keys[name], keep=False).sum()),
        "Distinct dates": len(dates),
        "First date": first.date(),
        "Last date": last.date(),
        "Calendar dates absent": len(absent_dates[name]) if name in absent_dates else "Event dates only",
    })

display(pd.DataFrame(inventory).set_index("Source"))
display(pd.DataFrame({
    name: dates.to_series().groupby(dates.year).size()
    for name, dates in absent_dates.items()
}).fillna(0).astype(int).rename_axis("Year").add_suffix(" dates absent"))
display(kiel_week.groupby(kiel_week["Datum"].dt.year).size().rename("Kiel Week dates").to_frame())

expected_ids = sales["Datum"].dt.strftime("%y%m%d") + sales["Warengruppe"].astype(str)
print(f"Sales IDs present: {sales['id'].notna().sum():,}")
print(f"Rows with repeated sales IDs: {sales['id'].duplicated(keep=False).sum():,}")
print(f"Sales IDs matching the date and product group: {sales['id'].eq(expected_ids).sum():,}")

## Column types, ranges, and missing values

The table counts blank values in each source column. The following comparisons show whether weather fields are missing together and whether the missing values cluster in particular years. For the source’s plausibility bounds, cloud cover is 0–8 octas, wind speed is nonnegative, and weather codes are whole numbers from 0 to 99.

In [ ]:
column_profiles = []
for name, frame in frames.items():
    for column in frame.columns:
        values = frame[column]
        valid = values.dropna()
        minimum = valid.min() if len(valid) else "—"
        maximum = valid.max() if len(valid) else "—"
        if column == "Datum" and len(valid):
            minimum, maximum = minimum.date(), maximum.date()
        column_profiles.append({
            "Source": name,
            "Column": column,
            "Type": str(values.dtype),
            "Missing": int(values.isna().sum()),
            "Minimum": minimum,
            "Maximum": maximum,
        })

display(pd.DataFrame(column_profiles).set_index(["Source", "Column"]))
weather_fields = ["Bewoelkung", "Temperatur", "Windgeschwindigkeit", "Wettercode"]
weather_missing = weather[weather_fields].isna()
missing_by_year = weather_missing.groupby(weather["Datum"].dt.year).sum()
observations_by_year = weather.groupby(weather["Datum"].dt.year).size()
display(missing_by_year.assign(Rows=observations_by_year))
display(weather_missing.value_counts().rename("Rows").to_frame())

code_summary = (
    weather.assign(weather_code_missing=weather["Wettercode"].isna())
    .groupby("weather_code_missing")
    .agg(
        rows=("Datum", "size"),
        cloud_missing=("Bewoelkung", lambda values: values.isna().sum()),
        median_temperature=("Temperatur", "median"),
        median_wind_speed=("Windgeschwindigkeit", "median"),
    )
)
display(code_summary.round(2))

codes = weather["Wettercode"].dropna()
display(pd.Series({
    "Nonpositive turnover": sales["Umsatz"].le(0).sum(),
    "Unexpected product group": (~sales["Warengruppe"].isin(product_names)).sum(),
    "Cloud cover outside 0–8": (~weather["Bewoelkung"].dropna().between(0, 8)).sum(),
    "Negative wind speed": weather["Windgeschwindigkeit"].lt(0).sum(),
    "Weather code outside 0–99 or fractional": (~codes.between(0, 99) | codes.mod(1).ne(0)).sum(),
    "Unexpected Kiel Week flag": kiel_week["KielerWoche"].ne(1).sum(),
}, name="Rows").to_frame())

## Sales availability by product group

An opening date in this profile has at least one recorded sale. A blank cell and an absent product-date row are different observations. Seasonal bread uses the October–January window in the canvas. For a first look at the other groups, all recorded opening dates are counted as possible dates.

In [ ]:
sales_dates = pd.DatetimeIndex(sales["Datum"].unique())
seasonal_months = [10, 11, 12, 1]
seasonal_dates = sales_dates[sales_dates.month.isin(seasonal_months)]

product_coverage = sales.groupby("Warengruppe").agg(
    rows=("Datum", "size"),
    dates_with_sales=("Datum", "nunique"),
    first_sale=("Datum", "min"),
    last_sale=("Datum", "max"),
    min_turnover=("Umsatz", "min"),
    median_turnover=("Umsatz", "median"),
    max_turnover=("Umsatz", "max"),
)
product_coverage["possible_open_dates"] = [
    len(seasonal_dates) if group == 6 else len(sales_dates) for group in product_coverage.index
]
product_coverage["open_dates_without_row"] = (
    product_coverage["possible_open_dates"] - product_coverage["dates_with_sales"]
)
product_coverage["product"] = product_coverage.index.map(product_names)
turnover_precision = {"min_turnover": 2, "median_turnover": 2, "max_turnover": 2}
display(product_coverage.round(turnover_precision))

confectionery_dates = sales.loc[sales["Warengruppe"].eq(4), "Datum"]
confectionery_gaps = sales_dates.difference(confectionery_dates)
gaps_by_weekday = pd.Series(confectionery_gaps.day_name()).value_counts()
display(gaps_by_weekday.reindex(weekdays, fill_value=0).rename("Confectionery dates absent"))

bread_dates = pd.DatetimeIndex(sales.loc[sales["Warengruppe"].eq(6), "Datum"])
bread_season_dates = bread_dates[bread_dates.month.isin(seasonal_months)]
print(f"Seasonal bread rows outside October–January: {len(bread_dates) - len(bread_season_dates):,}")
print(f"Seasonal bread rows in January: {(bread_dates.month == 1).sum():,}")


def counts_by_season(dates):
    start_years = dates.year - (dates.month == 1)
    return pd.Series(start_years).value_counts().sort_index()


display(pd.DataFrame({
    "October–January opening dates": counts_by_season(seasonal_dates),
    "Seasonal bread dates": counts_by_season(bread_season_dates),
}).rename_axis("Season start year"))
display(pd.Series(absent_dates["Sales"].strftime("%Y-%m-%d"), name="Dates without any sales").to_frame())
display(sales.nlargest(5, "Umsatz")[["Datum", "Warengruppe", "Umsatz"]])

## Potential joins on date

The match counts use unique dates on each side. A left join starts with the sales table and keeps its records. An inner join keeps only rows whose date also occurs in the other file. The weather and Kiel Week dates must be unique for the proposed many-to-one joins.

In [ ]:
join_results = []
for name, frame in {"Weather": weather, "Kiel Week": kiel_week}.items():
    source_dates = pd.DatetimeIndex(frame["Datum"].unique())
    shared_dates = sales_dates.intersection(source_dates)
    left_join = sales.merge(frame, on="Datum", how="left", validate="many_to_one")
    inner_join = sales.merge(frame, on="Datum", how="inner", validate="many_to_one")
    join_results.append({
        "Source": name,
        "Shared dates": len(shared_dates),
        "Sales dates matched (%)": 100 * len(shared_dates) / len(sales_dates),
        "Source dates matched (%)": 100 * len(shared_dates) / len(source_dates),
        "Left join rows": len(left_join),
        "Inner join rows": len(inner_join),
        "Sales rows without source date": len(left_join) - len(inner_join),
    })
    if name == "Weather":
        print("Opening dates without a weather row:", sales_dates.difference(source_dates).tolist())
    else:
        display(
            kiel_week.groupby(kiel_week["Datum"].dt.year)["Datum"]
            .agg(event_dates="size", sales_dates=lambda dates: dates.isin(sales_dates).sum())
        )

display(pd.DataFrame(join_results).set_index("Source").round(2))
combined = (
    sales.merge(weather, on="Datum", how="left", validate="many_to_one")
    .merge(kiel_week, on="Datum", how="left", validate="many_to_one")
)
print(f"Sales rows after both left joins: {len(combined):,}")

## Turnover distributions and weather correlations

The plots use the approved training dates, 1 July 2013 through 31 July 2017. The box plots show how turnover varies within each product group. The weekday plots compare median turnover on opening days, ordered Monday through Sunday. Correlations describe how historical turnover and measured weather move together within a product group. Each correlation uses the dates on which both values were recorded. Observed weather becomes available after the planning day and is reserved for the team’s upper-limit experiment.

In [ ]:
training = combined.loc[combined["Datum"].between("2013-07-01", "2017-07-31")].copy()
training["Product"] = training["Warengruppe"].map(product_names)
training["Weekday"] = training["Datum"].dt.day_name()

fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")
training.boxplot(column="Umsatz", by="Warengruppe", ax=ax, grid=False)
fig.suptitle("")
ax.set_title("Daily turnover by product group")
ax.set_xticks(range(1, len(product_names) + 1), list(product_names.values()))
ax.set(xlabel="", ylabel="Turnover")
plt.show()

weekday_medians = (
    training.groupby(["Weekday", "Product"])["Umsatz"]
    .median().unstack("Product").reindex(index=weekdays, columns=list(product_names.values()))
)
weekday_medians.plot(
    subplots=True, layout=(2, 3), marker="o", legend=False,
    sharex=True, figsize=(13, 7), rot=25, ylabel="Median turnover",
    xticks=list(range(len(weekdays))),
    title=list(product_names.values()),
)
plt.suptitle("Median turnover by weekday · training period")
plt.tight_layout()
plt.show()

continuous_weather = ["Bewoelkung", "Temperatur", "Windgeschwindigkeit"]
correlations = []
for product, group_sales in training.groupby("Product", sort=False):
    for column in continuous_weather:
        matched = group_sales[["Umsatz", column]].dropna()
        correlations.append({
            "Product": product,
            "Weather measure": column,
            "Paired rows": len(matched),
            "Pearson r": matched["Umsatz"].corr(matched[column]),
        })

display(pd.DataFrame(correlations).set_index(["Product", "Weather measure"]).round(2))